In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

In [66]:
import pandas as pd

# ---- Dataset A: y = x1*x2*x3*x4 ----
df_prod = pd.DataFrame({
    "year": [2022, 2023, 2024, 2025],
    "x1":   [100_000, 106_000, 111_000, 118_000],   # customers
    "x2":   [1.30, 1.32, 1.31, 1.35],               # policies per customer
    "x3":   [0.040, 0.043, 0.041, 0.046],           # claims per policy
    "x4":   [28_000, 29_500, 31_000, 32_500],       # average claim amount
})
df_prod["y"] = df_prod.x1 * df_prod.x2 * df_prod.x3 * df_prod.x4
df_prod = df_prod[["year", "y", "x1", "x2", "x3", "x4"]].set_index("year")

# ---- Dataset B: y = x1+x2+x3+x4 (same units: claim amount in millions) ----
df_sum = pd.DataFrame({
    "year": [2022, 2023, 2024, 2025],
    "x1":   [432.0, 490.0, 502.6, 614.3],           # Term Life claims
    "x2":   [165.0, 165.2, 196.3, 201.5],           # Whole Life claims
    "x3":   [336.0, 390.6, 474.0, 551.8],           # Health claims
    "x4":   [ 40.0,  42.0,  41.0,  47.0],           # Other / riders claims
})
df_sum["y"] = df_sum.x1 + df_sum.x2 + df_sum.x3 + df_sum.x4
df_sum = df_sum[["year", "y", "x1", "x2", "x3", "x4"]].set_index("year")

factors = ['x1', 'x2', 'x3', 'x4']

In [ ]:
import numpy as np

class LMDI:
    """Log Mean Divisia Index (LMDI-I) decomposition for one pair of periods.

    Splits the change of a total ``y`` between an earlier period ``b`` and a later
    period ``a`` into the effect of each factor, in two forms:

    * additive: amounts that add up to ``y_a - y_b``.
    * multiplicative: growth factors that multiply to ``y_a / y_b``.

    Two kinds of ``y`` are supported. What you pass as the cell value
    (``a_tot``, ``b_tot``) and the grand total (``y_a``, ``y_b``) depends on the kind:

    ====================  =====================  ====================
    y                     a_tot, b_tot (cell)    y_a, y_b (total)
    ====================  =====================  ====================
    x1 * x2 * ... * xn    a.prod(), b.prod()     omit (weight is 1)
    x1 + x2 + ... + xn    a, b                   a.sum(), b.sum()
    ====================  =====================  ====================

    Example:
        >>> lmdi = LMDI()
        >>> # y is a product
        >>> add, mult = lmdi.run(a, b, a.prod(), b.prod())
        >>> # y is a sum
        >>> add, mult = lmdi.run(a, b, a, b, a.sum(), b.sum())
    """

    def __init__(self): ...

    def log_mean(self, a, b):
        """Logarithmic mean L(a, b) = (a - b) / (ln a - ln b).

        Args:
            a: Value at the later period (scalar or array, must be > 0).
            b: Value at the earlier period (scalar or array, must be > 0).

        Returns:
            L(a, b). When ``a`` is close to ``b`` the limit L(a, a) = a is
            returned instead, to avoid dividing by zero.
        """
        a, b = np.asarray(a, float), np.asarray(b, float)
        return np.where(np.isclose(a, b), a, (a - b) / (np.log(a) - np.log(b)))

    def log_change(self, a, b):
        """Log change ln(a / b), the growth of a factor in log terms.

        Args:
            a: Factor values at the later period.
            b: Factor values at the earlier period.

        Returns:
            ln(a / b), same shape as the inputs.
        """
        return np.log(a / b)

    def additive(self, a, b, a_tot, b_tot):
        """Additive effect of each factor: L(cell) * ln(factor ratio).

        The effects add up to the change in the total (y_a - y_b).

        Args:
            a: Factor values at the later period.
            b: Factor values at the earlier period.
            a_tot: Value of the cell the factor lives in, later period.
                Product: ``a.prod()``. Sum: ``a`` (each term is its own cell).
            b_tot: Same as ``a_tot``, earlier period.

        Returns:
            Effect of each factor, in the same units as y.
        """
        return self.log_mean(a_tot, b_tot) * self.log_change(a, b)

    def multiplicative(self, a, b, a_tot, b_tot, y_a=None, y_b=None):
        """Multiplicative effect of each factor: exp(L(cell) / L(total) * ln(ratio)).

        The effects multiply to the growth of the total (y_a / y_b).

        Args:
            a: Factor values at the later period.
            b: Factor values at the earlier period.
            a_tot: Cell value, later period (see ``additive``).
            b_tot: Cell value, earlier period.
            y_a: Grand total, later period. Product: leave as None (the cell is
                the whole total, so the weight is 1). Sum: ``a.sum()``.
            y_b: Grand total, earlier period. Same rule as ``y_a``.

        Returns:
            Growth factor attributed to each factor.
        """
        y_a = a_tot if y_a is None else y_a
        y_b = b_tot if y_b is None else y_b
        weights = self.log_mean(a_tot, b_tot) / self.log_mean(y_a, y_b)
        return np.exp(weights * self.log_change(a, b))

    def run(self, a, b, a_tot, b_tot, y_a=None, y_b=None):
        """Compute both the additive and multiplicative effects.

        Args:
            a: Factor values at the later period.
            b: Factor values at the earlier period.
            a_tot: Cell value, later period.
            b_tot: Cell value, earlier period.
            y_a: Grand total, later period (only needed when y is a sum).
            y_b: Grand total, earlier period (only needed when y is a sum).

        Returns:
            A tuple ``(additive, multiplicative)``.
        """
        additive = self.additive(a, b, a_tot, b_tot)
        multiplicative = self.multiplicative(a, b, a_tot, b_tot, y_a, y_b)
        return additive, multiplicative


def compare(additive, multiplicative, diff, growth):
    """Check that the effects add up to the totals.

    Args:
        additive: Additive effects from ``LMDI.run``.
        multiplicative: Multiplicative effects from ``LMDI.run``.
        diff: Change in the total, y_a - y_b (use the product or the sum).
        growth: Growth of the total, y_a / y_b.

    Raises:
        AssertionError: If the additive effects do not sum to ``diff`` or the
            multiplicative effects do not multiply to ``growth``.
    """
    assert np.isclose(additive.sum(), diff), f'additive failed: {additive.sum():.6g} vs {diff:.6g}'
    assert np.isclose(multiplicative.prod(), growth), f'multiplicative failed: {multiplicative.prod():.6g} vs {growth:.6g}'

In [101]:
factors = ["x1", "x2", "x3", "x4"]

def test_product(df, t0, t1):
    """y = x1 * x2 * x3 * x4: one cell, the whole product."""
    a = df.loc[t1, factors].values
    b = df.loc[t0, factors].values
    a_tot, b_tot = a.prod(), b.prod()
    additive, multiplicative = LMDI().run(a, b, a_tot, b_tot)
    compare(additive, multiplicative, a_tot - b_tot, a_tot / b_tot)

def test_sum(df, t0, t1):
    """y = x1 + x2 + x3 + x4: each term is its own cell."""
    a = df.loc[t1, factors].values
    b = df.loc[t0, factors].values
    y_a, y_b = a.sum(), b.sum()
    additive, multiplicative = LMDI().run(a, b, a, b, y_a, y_b)
    compare(additive, multiplicative, y_a - y_b, y_a / y_b)

def test_all(df, test, name):
    years = list(df.index)
    for t0, t1 in zip(years[:-1], years[1:]):
        test(df, t0, t1)
        print(f"{name}: {t0}->{t1} OK")

test_all(df_prod, test_product, "product")
test_all(df_sum, test_sum, "sum")

product: 2022->2023 OK
product: 2023->2024 OK
product: 2024->2025 OK
sum: 2022->2023 OK
sum: 2023->2024 OK
sum: 2024->2025 OK


In [102]:
rows = [
    # year, product,    x1,     x2,   x3,     x4
    (2022, "Term Life", 60_000, 1.20, 0.0120, 500_000),
    (2022, "Health",    80_000, 1.50, 0.3500,   8_000),
    (2023, "Term Life", 63_000, 1.22, 0.0125, 510_000),
    (2023, "Health",    86_000, 1.52, 0.3600,   8_300),
]
df = pd.DataFrame(rows, columns=["year", "product", "x1", "x2", "x3", "x4"])
df["y"] = df.x1 * df.x2 * df.x3 * df.x4          # claim amount of each cell
df = df.set_index(["year", "product"])
factors = ["x1", "x2", "x3", "x4"]

In [103]:
d0, d1 = df.loc[2022], df.loc[2023]            # rows = product
y_b, y_a = d0.y.sum(), d1.y.sum()              # grand totals
lmdi = LMDI()

add_rows, mult_rows = {}, {}
for p in d1.index:
    a, b = d1.loc[p, factors].values, d0.loc[p, factors].values
    add_rows[p], mult_rows[p] = lmdi.run(a, b, a.prod(), b.prod(), y_a, y_b)

add  = pd.DataFrame(add_rows,  index=factors).T
mult = pd.DataFrame(mult_rows, index=factors).T
add.loc["TOTAL"]  = add.sum()                  # additive: sum over cells
mult.loc["TOTAL"] = mult.prod()                # multiplicative: product over cells

compare(add.loc["TOTAL"], mult.loc["TOTAL"], y_a - y_b, y_a / y_b)

In [109]:
df

x1    x2      x3      x4            y
year product                                            
2022 Term Life  60000  1.20  0.0120  500000  432000000.0
     Health     80000  1.50  0.3500    8000  336000000.0
2023 Term Life  63000  1.22  0.0125  510000  489982500.0
     Health     86000  1.52  0.3600    8300  390591360.0

In [114]:
df.loc[2023].y.sum() - df.loc[2022].y.sum()

np.float64(112573860.0)

In [111]:
add.assign(total=add.sum(axis=1)).round(2)

,x1,x2,x3,x4,total
Term Life,22462155.55,7609807.38,18793746.78,9116790.28,57982500.0
Health,26224270.39,4802865.46,10215071.03,13349153.12,54591360.0
TOTAL,48686425.94,12412672.85,29008817.81,22465943.41,112573860.0


In [115]:
df.loc[2023].y.sum() / df.loc[2022].y.sum()

np.float64(1.146580546875)

In [117]:
mult.assign(total=mult.prod(axis=1)).round(4)

,x1,x2,x3,x4,total
Term Life,1.0277,1.0093,1.0231,1.0111,1.0730
Health,1.0324,1.0059,1.0125,1.0164,1.0686
TOTAL,1.0609,1.0152,1.0359,1.0277,1.1466
